# Corpus Visualizer — mod/legislativa
**Tenant:** `scriptorium` · **Database:** `mod-legislativa`

Proyecta los embeddings de las 5 colecciones Chroma locales en un espacio 2D/3D usando UMAP.  
Cada punto es una pieza del lore. El color codifica la colección de origen.

| Colección | Piezas |
|-----------|--------|
| `ml_personajes` | P-01…P-09 |
| `ml_eventos` | T-01…T-14 |
| `ml_recursos` | R-01…R-10 |
| `ml_piezas_media` | S-01…S-13 + N-01…N-05 |
| `ml_hilo_narrativo` | §0-§4 (fragmentos sintéticos) |

**Requisitos:** `pip install chromadb umap-learn plotly pandas numpy scikit-learn`

In [11]:
%pip install -q chromadb umap-learn plotly pandas numpy scikit-learn "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -q nbformat

Note: you may need to restart the kernel to use updated packages.


In [16]:
import chromadb
import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.preprocessing import LabelEncoder

def show(fig):
    """Renderiza un figura Plotly en VS Code sin depender de nbformat."""
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

# ── Conexión al cliente Chroma local persistente ────────────────────────────
STORAGE_PATH = r"C:\Users\aleph\OASIS\aleph-scriptorium\ARCHIVO\PLUGINS\VECTOR_MACHINE\STORAGE"

client = chromadb.PersistentClient(path=STORAGE_PATH)
print("Colecciones disponibles:", [c.name for c in client.list_collections()])

Colecciones disponibles: ['ml_hilo_narrativo', 'ml_piezas_media', 'ml_recursos', 'ml_eventos', 'ml_personajes']


## Carga de embeddings desde las 5 colecciones

In [7]:
COLLECTIONS = [
    "ml_personajes",
    "ml_eventos",
    "ml_recursos",
    "ml_piezas_media",
    "ml_hilo_narrativo",
]

# Etiqueta corta para leyendas
LABELS = {
    "ml_personajes":     "Personajes [P-*]",
    "ml_eventos":        "Eventos [T-*]",
    "ml_recursos":       "Recursos [R-*]",
    "ml_piezas_media":   "Media [S-*/N-*]",
    "ml_hilo_narrativo": "Hilo [HN-*]",
}

rows = []
for col_name in COLLECTIONS:
    col = client.get_collection(col_name)
    result = col.get(include=["embeddings", "documents", "metadatas"])
    for doc_id, emb, doc, meta in zip(
        result["ids"], result["embeddings"], result["documents"], result["metadatas"]
    ):
        rows.append({
            "id":         doc_id,
            "coleccion":  LABELS[col_name],
            "col_raw":    col_name,
            "bloque":     meta.get("bloque", "?"),
            "tipo":       meta.get("tipo", meta.get("subtipo", "?")),
            "marca":      meta.get("marca", doc_id),
            "texto":      doc[:120] + "…" if len(doc) > 120 else doc,
            "embedding":  emb,
        })

df = pd.DataFrame(rows)
embeddings_matrix = np.array(df["embedding"].tolist())

print(f"Total piezas cargadas: {len(df)}")
print(df.groupby("coleccion").size().to_string())

Total piezas cargadas: 64
coleccion
Eventos [T-*]       14
Hilo [HN-*]         16
Media [S-*/N-*]     15
Personajes [P-*]     9
Recursos [R-*]      10


## Proyección UMAP — 2D y 3D

`n_neighbors=5` adecuado para corpus pequeño (64 puntos).  
`min_dist=0.2` para que los clusters no colapsen.

In [9]:
UMAP_PARAMS = dict(n_neighbors=5, min_dist=0.2, random_state=42, low_memory=False)

reducer_2d = umap.UMAP(n_components=2, **UMAP_PARAMS)
proj_2d = reducer_2d.fit_transform(embeddings_matrix)
df["x"] = proj_2d[:, 0]
df["y"] = proj_2d[:, 1]

reducer_3d = umap.UMAP(n_components=3, **UMAP_PARAMS)
proj_3d = reducer_3d.fit_transform(embeddings_matrix)
df["x3"] = proj_3d[:, 0]
df["y3"] = proj_3d[:, 1]
df["z3"] = proj_3d[:, 2]

print("Proyección 2D shape:", proj_2d.shape)
print("Proyección 3D shape:", proj_3d.shape)

Proyección 2D shape: (64, 2)
Proyección 3D shape: (64, 3)


c:\Users\aleph\OASIS\aleph-scriptorium\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\aleph\OASIS\aleph-scriptorium\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Vista 2D — por colección

In [18]:
COLOR_MAP = {
    "Personajes [P-*]":  "#e45756",
    "Eventos [T-*]":     "#4c78a8",
    "Recursos [R-*]":    "#72b7b2",
    "Media [S-*/N-*]":   "#f58518",
    "Hilo [HN-*]":       "#b279a2",
}

fig2d = px.scatter(
    df,
    x="x", y="y",
    color="coleccion",
    color_discrete_map=COLOR_MAP,
    hover_data={"marca": True, "tipo": True, "texto": True, "x": False, "y": False},
    text="marca",
    title="mod/legislativa — Espacio de embeddings 2D (UMAP)",
    width=900, height=650,
)
fig2d.update_traces(
    textposition="top center",
    textfont_size=9,
    marker=dict(size=10, opacity=0.85),
)
fig2d.update_layout(
    legend_title_text="Colección",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    font_color="#e0e0e0",
)
show(fig2d)

## Vista 3D — por colección

In [19]:
fig3d = px.scatter_3d(
    df,
    x="x3", y="y3", z="z3",
    color="coleccion",
    color_discrete_map=COLOR_MAP,
    hover_data={"marca": True, "tipo": True, "texto": True,
                "x3": False, "y3": False, "z3": False},
    text="marca",
    title="mod/legislativa — Espacio de embeddings 3D (UMAP)",
    width=900, height=700,
)
fig3d.update_traces(
    textposition="top center",
    textfont_size=8,
    marker=dict(size=5, opacity=0.9),
)
fig3d.update_layout(
    legend_title_text="Colección",
    paper_bgcolor="#1a1a2e",
    font_color="#e0e0e0",
    scene=dict(
        bgcolor="#1a1a2e",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)
show(fig3d)

## Análisis de clusters automático (KMeans)

Detecta agrupaciones que el UMAP revela geométricamente,  
independientemente de las colecciones asignadas por el Archivero.

In [20]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Buscar el K óptimo por silhouette score en rango 2-8
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_matrix)
    scores[k] = silhouette_score(embeddings_matrix, labels)

best_k = max(scores, key=scores.get)
print(f"Silhouette scores: {scores}")
print(f"K óptimo: {best_k} (score={scores[best_k]:.3f})")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(embeddings_matrix).astype(str)

fig_cluster = px.scatter(
    df,
    x="x", y="y",
    color="cluster",
    symbol="coleccion",
    hover_data={"marca": True, "tipo": True, "texto": True,
                "coleccion": True, "x": False, "y": False},
    text="marca",
    title=f"Clusters semánticos automáticos (K={best_k}) sobre espacio 2D",
    width=900, height=650,
)
fig_cluster.update_traces(
    textposition="top center",
    textfont_size=8,
    marker=dict(size=10, opacity=0.85),
)
fig_cluster.update_layout(
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    font_color="#e0e0e0",
)
show(fig_cluster)

# Mostrar composición de cada cluster
print("\nComposición de clusters:")
print(df.groupby(["cluster", "coleccion"])["marca"].apply(list).to_string())

Silhouette scores: {2: 0.04784317686398234, 3: 0.04943959895752674, 4: 0.05740032122021927, 5: 0.07714326766178439, 6: 0.0627891994007802, 7: 0.08095898229721418, 8: 0.0792342714957057}
K óptimo: 7 (score=0.081)



Composición de clusters:
cluster  coleccion       
0        Eventos [T-*]                                                [T-05]
         Hilo [HN-*]                                                [HN-S4f]
         Media [S-*/N-*]                                        [S-11, S-13]
         Personajes [P-*]                                             [P-06]
1        Eventos [T-*]                        [T-03, T-04, T-06, T-09, T-13]
         Hilo [HN-*]                                         [HN-S1e, HN-S2]
         Media [S-*/N-*]                                  [S-01, N-02, N-05]
         Personajes [P-*]                                 [P-01, P-05, P-09]
         Recursos [R-*]                                               [R-07]
2        Hilo [HN-*]                                [HN-S4a, HN-S4b, HN-S4e]
         Media [S-*/N-*]                                  [S-03, S-04, N-04]
         Personajes [P-*]                                             [P-04]
3        Eventos [T-*]  

## Visualización de query — dónde aterriza una pregunta nueva

Proyecta el embedding de un texto libre en el espacio del corpus.  
Útil para ver si una query cae dentro de un cluster existente (CONFIRMA espacial)  
o en una zona vacía (posible ausencia estructural).

In [21]:
from chromadb.utils import embedding_functions

# ── Embedding de la query usando la misma función default (all-MiniLM-L6-v2) ─
QUERY = "¿Cuál es la estrategia de lucro indirecto que usa la acusación?"

ef = embedding_functions.DefaultEmbeddingFunction()
query_emb = np.array(ef([QUERY]))

# Proyectar en el espacio 2D ya ajustado (transform, no fit_transform)
query_2d = reducer_2d.transform(query_emb)

# Recuperar los 5 más cercanos por distancia euclidiana en espacio proyectado
dists = np.linalg.norm(proj_2d - query_2d, axis=1)
df["dist_to_query"] = dists
top5 = df.nsmallest(5, "dist_to_query")[["marca", "coleccion", "tipo", "texto", "dist_to_query"]]
print(f"Query: '{QUERY}'\n")
print("Top 5 piezas más cercanas en espacio 2D:")
print(top5.to_string(index=False))

# Visualización: corpus + punto de query
df_query = pd.DataFrame([{
    "x": query_2d[0, 0], "y": query_2d[0, 1],
    "marca": "◉ QUERY", "coleccion": "Query", "tipo": "query",
    "texto": QUERY, "cluster": "q"
}])
df_plot = pd.concat([df, df_query], ignore_index=True)

color_map_q = {**COLOR_MAP, "Query": "#ffffff"}

fig_q = px.scatter(
    df_plot, x="x", y="y",
    color="coleccion",
    color_discrete_map=color_map_q,
    hover_data={"marca": True, "tipo": True, "texto": True, "x": False, "y": False},
    text="marca",
    title="Query proyectada en el corpus",
    width=900, height=650,
)
fig_q.update_traces(
    textposition="top center",
    textfont_size=8,
    marker=dict(size=9, opacity=0.85),
)
# Resaltar el punto de la query
fig_q.update_traces(
    selector={"name": "Query"},
    marker=dict(size=16, symbol="star", color="white", line=dict(color="red", width=2)),
    textfont=dict(size=11, color="white"),
)
fig_q.update_layout(
    plot_bgcolor="#1a1a2e", paper_bgcolor="#1a1a2e", font_color="#e0e0e0"
)
show(fig_q)

Query: '¿Cuál es la estrategia de lucro indirecto que usa la acusación?'

Top 5 piezas más cercanas en espacio 2D:
marca       coleccion          tipo                                                                                                                     texto  dist_to_query
 T-11   Eventos [T-*] procedimiento [T-11] La demora de 4 años. Instrucción larga entre la denuncia y la vista oral. El tiempo procesal es en sí mismo casti…       0.585279
 T-10   Eventos [T-*] procedimiento [T-10] LA CAUSA. El entramado de Cerezo interpone acción penal contra Feo. EGEDA, Mercury Films, FlixOlé contra un archi…       0.695193
 N-01 Media [S-*/N-*]       noticia [N-01] Dato contextual: el propio Cerezo fue condenado en primera instancia como cooperador necesario de apropiación ind…       0.697364
 T-12   Eventos [T-*]        juicio [T-12] El juicio. Vista oral, 9 de abril de 2026. David Bravo asume la defensa de Feo. El letrado que defendió a Pablo S…       0.855237
 S-10 Media [S-*/N-*

## Divergencia corpus.md ↔ geometría

Detecta piezas que **el Archivero clasificó en una colección** pero que geométricamente  
son más cercanas a piezas de otra colección — posible tensión semántica no capturada en texto.

In [22]:
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=4, metric="cosine").fit(embeddings_matrix)
distances, indices = nbrs.kneighbors(embeddings_matrix)

divergencias = []
for i, (dists_i, idxs_i) in enumerate(zip(distances, indices)):
    pieza = df.iloc[i]
    vecinos = df.iloc[idxs_i[1:]]  # excluir el propio punto
    # Si la mayoría de sus vecinos son de otra colección → divergencia
    vecinos_col = vecinos["col_raw"].value_counts()
    col_mayoritaria = vecinos_col.index[0]
    if col_mayoritaria != pieza["col_raw"] and vecinos_col.iloc[0] >= 2:
        divergencias.append({
            "marca":          pieza["marca"],
            "col_asignada":   pieza["coleccion"],
            "col_geometrica": LABELS[col_mayoritaria],
            "vecinos":        list(vecinos["marca"]),
            "dist_media":     round(float(dists_i[1:].mean()), 4),
        })

if divergencias:
    print(f"Piezas con divergencia corpus ↔ geometría ({len(divergencias)}):\n")
    for d in divergencias:
        print(f"  {d['marca']}")
        print(f"    Asignada:   {d['col_asignada']}")
        print(f"    Geometría:  {d['col_geometrica']}")
        print(f"    Vecinos:    {d['vecinos']}")
        print(f"    Dist media: {d['dist_media']}\n")
else:
    print("Sin divergencias detectadas — la taxonomía del Archivero coincide con la geometría.")

Piezas con divergencia corpus ↔ geometría (27):

  P-05
    Asignada:   Personajes [P-*]
    Geometría:  Media [S-*/N-*]
    Vecinos:    ['N-05', 'S-01', 'P-01']
    Dist media: 0.2782

  P-07
    Asignada:   Personajes [P-*]
    Geometría:  Hilo [HN-*]
    Vecinos:    ['HN-S4d', 'P-02', 'HN-S1a']
    Dist media: 0.3291

  P-08
    Asignada:   Personajes [P-*]
    Geometría:  Media [S-*/N-*]
    Vecinos:    ['S-05', 'T-08', 'N-03']
    Dist media: 0.3902

  T-06
    Asignada:   Eventos [T-*]
    Geometría:  Media [S-*/N-*]
    Vecinos:    ['S-01', 'P-01', 'N-05']
    Dist media: 0.2788

  T-09
    Asignada:   Eventos [T-*]
    Geometría:  Media [S-*/N-*]
    Vecinos:    ['N-05', 'P-04', 'S-01']
    Dist media: 0.3321

  T-13
    Asignada:   Eventos [T-*]
    Geometría:  Hilo [HN-*]
    Vecinos:    ['HN-S2', 'T-05', 'HN-S3']
    Dist media: 0.4442

  R-02
    Asignada:   Recursos [R-*]
    Geometría:  Hilo [HN-*]
    Vecinos:    ['HN-S0', 'T-08', 'HN-S1a']
    Dist media: 0.398

  R-03


## Exportar a GH Pages — `docs/legislativa/cuadernos/`

Genera los HTMLs estáticos interactivos para la página de cuadernos.

In [ ]:
import os
import yaml
from datetime import date

EXPORT_DIR = r"C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\legislativa\cuadernos"
DATA_FILE  = r"C:\Users\aleph\OASIS\aleph-scriptorium\DocumentMachineSDK\docs\_data\cuadernos_legislativa.yml"
os.makedirs(EXPORT_DIR, exist_ok=True)

exports = {
    "corpus_2d": {
        "fig":         fig2d,
        "title":       "Espacio de embeddings 2D",
        "descripcion": "Proyección UMAP 2D de las 5 colecciones (Personajes, Eventos, Recursos, Media, Hilo narrativo).",
    },
    "corpus_3d": {
        "fig":         fig3d,
        "title":       "Espacio de embeddings 3D",
        "descripcion": "Proyección UMAP 3D rotable. Permite ver la separación entre clusters de personajes, eventos y recursos.",
    },
    "corpus_clusters": {
        "fig":         fig_cluster,
        "title":       f"Clusters semánticos (K={best_k})",
        "descripcion": f"Agrupación K-means (K={best_k}) sobre el espacio vectorial. Revela si la taxonomía del Archivero coincide con la geometría.",
    },
    "corpus_query": {
        "fig":         fig_q,
        "title":       "Visualización de query",
        "descripcion": f"Proyección del embedding de «{QUERY}» en el espacio del corpus. Vecinos más cercanos resaltados.",
    },
}

# ── Exportar HTML ─────────────────────────────────────────────────────────────
for cuaderno_id, info in exports.items():
    fname = f"{cuaderno_id}.html"
    dest  = os.path.join(EXPORT_DIR, fname)
    info["fig"].write_html(dest, include_plotlyjs="cdn", full_html=True)
    print(f"✓ {fname}  ({os.path.getsize(dest) // 1024} KB)")

# ── Actualizar _data/cuadernos_legislativa.yml ────────────────────────────────
colecciones_str = " · ".join(LABELS.values())
hoy = date.today().isoformat()

registros = []
for cuaderno_id, info in exports.items():
    registros.append({
        "id":          cuaderno_id,
        "title":       info["title"],
        "file":        f"legislativa/cuadernos/{cuaderno_id}.html",
        "descripcion": info["descripcion"],
        "fecha":       hoy,
        "colecciones": colecciones_str,
    })

header = (
    "# Cuadernos vectoriales — mod/legislativa\n"
    "# Generado/actualizado automáticamente por la celda de exportación del notebook.\n"
    "# Cada entrada se convierte en una card en el catálogo.\n\n"
)
with open(DATA_FILE, "w", encoding="utf-8") as f:
    f.write(header)
    yaml.dump(registros, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"\n✓ {DATA_FILE} actualizado ({len(registros)} cuadernos)")


✓ corpus_2d.html  (20 KB)
✓ corpus_3d.html  (21 KB)
✓ corpus_clusters.html  (33 KB)
✓ corpus_query.html  (21 KB)
